# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/7ayder-99/flyrank_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: This is a "which pages first" ranking question with an observed label (is_declining, defined the same way as the Week-4 baseline: impressions_mar < impressions_feb). Per the toolkit, ranking with an observed label calls for a classifier's predicted probability evaluated at precision@K — starting with Logistic Regression (readable) and comparing against Random Forest (stronger, if it earns its place).

Features: All strictly from February and earlier (dim_content static fields + Feb-month aggregates) — never from March, since March data defines the label. This mirrors the leakage lesson from ML-04: a feature that overlaps with how the label was built isn't a feature, it's the label in disguise.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
%pip -q install duckdb

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/7ayder-99/flyrank_internship"
REPO_DIR = "flyrank_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import duckdb, pandas as pd
con = duckdb.connect()
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL_FEB   = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')"
REL_MARCH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

page_month_sql = """
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position
    FROM {rel}
    GROUP BY client_hash_id, content_hash_id
"""

feb_df = con.sql(page_month_sql.format(rel=REL_FEB)).df()
mar_df = con.sql(page_month_sql.format(rel=REL_MARCH)).df()

pages = mar_df.merge(feb_df, on=["client_hash_id", "content_hash_id"],
                      suffixes=("_mar", "_feb"), how="inner")

# staleness signal from dim_content
content = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_updated_date,
           content_type, word_count, category_count, competition, backlinks
    FROM {DIM_CONTENT}
""").df()

pages = pages.merge(content, on=["client_hash_id", "content_hash_id"], how="left")

ANALYSIS_DATE = pd.Timestamp("2026-03-31")
pages["content_updated_date"] = pd.to_datetime(pages["content_updated_date"])
pages["days_since_update"] = (ANALYSIS_DATE - pages["content_updated_date"]).dt.days

# the label — same definition as the Week-4 baseline
pages["is_declining"] = (pages["impressions_mar"] < pages["impressions_feb"]).astype(int)

# Feb-only CTR (safe feature, no March data)
pages["ctr_feb"] = pages["clicks_feb"] / pages["impressions_feb"].replace(0, pd.NA)

print("Pages rebuilt:", pages.shape)
pages.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages rebuilt: (303572, 17)


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,avg_position_mar,impressions_feb,clicks_feb,avg_position_feb,content_updated_date,content_type,word_count,category_count,competition,backlinks,days_since_update,is_declining,ctr_feb
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,4270.0,7.0,5.805560,2026-07-06,keyword article,2123,0,0.00,<NA>,-97,0,0.001639
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,440.0,2.0,3.994887,2026-05-18,keyword article,<NA>,0,0.45,<NA>,-48,0,0.004545
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,5271.0,4.0,7.101588,2026-05-20,keyword article,2546,0,0.00,<NA>,-50,0,0.000759
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,6690.0,19.0,7.323989,2026-07-06,keyword article,2330,0,0.00,<NA>,-97,1,0.00284
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,24.0,0.0,17.722222,2026-05-18,keyword article,<NA>,0,1.00,<NA>,-48,0,0.0


In [3]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(pages, groups=pages["client_hash_id"]))

train_df = pages.iloc[train_idx].copy()
test_df  = pages.iloc[test_idx].copy()

print("Train pages:", len(train_df), "| Test pages:", len(test_df))
print("Train clients:", train_df['client_hash_id'].nunique(),
      "| Test clients:", test_df['client_hash_id'].nunique())
# confirm no client appears in both
overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print("Client overlap (should be 0):", len(overlap))

Train pages: 264134 | Test pages: 39438
Train clients: 40 | Test clients: 10
Client overlap (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
FEATURES = ["days_since_update", "impressions_feb", "avg_position_feb", "ctr_feb",
            "word_count", "category_count", "competition", "backlinks"]

for df in (train_df, test_df):
    df[FEATURES] = df[FEATURES].fillna(0)

X_train, y_train = train_df[FEATURES], train_df["is_declining"]
X_test,  y_test  = test_df[FEATURES],  test_df["is_declining"]

print("Base rate (test):", y_test.mean().round(4))

Base rate (test): 0.156


/tmp/ipykernel_7889/1598146100.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[FEATURES] = df[FEATURES].fillna(0)
/tmp/ipykernel_7889/1598146100.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[FEATURES] = df[FEATURES].fillna(0)


In [5]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# corrected baseline: rule uses ONLY Feb-known signals (no March, no label leakage)
test_df["baseline_score"] = (
    (test_df["days_since_update"] >= 180).astype(int) *
    (test_df["impressions_feb"] >= 100).astype(int) *
    test_df["impressions_feb"]
)

baseline_p50 = precision_at_k(test_df["baseline_score"], y_test, 50)
print("Baseline precision@50:", round(baseline_p50, 3))

Baseline precision@50: 0.02


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]
logreg_p50 = precision_at_k(logreg_scores, y_test, 50)
print("Logistic Regression precision@50:", round(logreg_p50, 3))

Logistic Regression precision@50: 0.64


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_scores, y_test, 50)
print("Random Forest precision@50:", round(rf_p50, 3))

Random Forest precision@50: 0.7


In [8]:
import pandas as pd

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "Baseline rule (Week 4, corrected)",
               "Logistic Regression", "Random Forest"],
    "precision@50": [round(y_test.mean(), 3), round(baseline_p50, 3),
                      round(logreg_p50, 3), round(rf_p50, 3)]
})
print(comparison.to_string(index=False))

                           method  precision@50
               Base rate (random)         0.156
Baseline rule (Week 4, corrected)         0.020
              Logistic Regression         0.640
                    Random Forest         0.700


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# --- Feature importance: Random Forest ---
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("=== Random Forest feature importances ===")
print(importances)

# --- Feature importance: Logistic Regression coefficients ---
coefs = pd.Series(logreg.coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("\n=== Logistic Regression coefficients (standardized) ===")
print(coefs)

# --- 3 concrete wrong cases from the Random Forest ---
test_df["rf_score"] = rf_scores
test_df["rf_pred"] = (rf_scores >= 0.5).astype(int)

wrong = test_df[test_df["rf_pred"] != test_df["is_declining"]]
print(f"\nTotal wrong: {len(wrong)} out of {len(test_df)}")

cols_to_show = ["content_hash_id", "is_declining", "rf_pred", "rf_score",
                "days_since_update", "impressions_feb", "avg_position_feb", "ctr_feb"]
print("\n=== 3 concrete wrong cases ===")
print(wrong[cols_to_show].sample(3, random_state=42).to_string(index=False))

=== Random Forest feature importances ===
impressions_feb      0.462871
avg_position_feb     0.336582
days_since_update    0.093610
word_count           0.062259
ctr_feb              0.028290
competition          0.007937
category_count       0.006521
backlinks            0.001930
dtype: float64

=== Logistic Regression coefficients (standardized) ===
avg_position_feb     0.437112
impressions_feb      0.204557
days_since_update    0.148808
ctr_feb              0.107313
word_count           0.055948
category_count       0.054445
backlinks           -0.051827
competition          0.027126
dtype: float64

Total wrong: 6292 out of 39438

=== 3 concrete wrong cases ===
         content_hash_id  is_declining  rf_pred  rf_score  days_since_update  impressions_feb  avg_position_feb  ctr_feb
content_496e3171a073ff6f             1        0  0.303416                -73            993.0          2.166057      0.0
content_e7fc0c0517e031db             1        0  0.304520                -78         

(46.3%) and average search position (33.7%), followed by days since update (9.4%). These features are logically related to content performance and provide a reasonable basis for predicting decline. Logistic Regression supported this finding, identifying average position, impressions, and days since update as the strongest predictors. Analysis of three misclassified cases showed that the model mainly struggled with borderline situations where the current performance metrics did not fully reflect the underlying decline status. This suggests that incorporating temporal change features, such as percentage change in impressions or position compared with the previous period, could help the model better capture actual content deterioration.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.